In [7]:
import pandas as pd
import json
import logging
from pathlib import Path
from typing import Optional, Dict, Any

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)


def filter_conversations_by_reward_threshold(input_csv_path: str, 
                                           output_csv_path: str,
                                           combined_reward_threshold: float = 0.7,
                                           save_format: str = "preference_pairs") -> Dict[str, Any]:
    """
    Filter preference pairs based on combined reward threshold
    
    Args:
        input_csv_path: Path to the input CSV file (preference_pairs_with_rewards_final.csv)
        output_csv_path: Path to save the filtered results
        combined_reward_threshold: Minimum combined reward threshold (default: 0.7)
        save_format: "preference_pairs" (keep chosen/rejected format) or "chosen_only" (only chosen conversations)
        
    Returns:
        Dictionary with filtering statistics
    """
    
    logger.info(f"Loading preference pairs from: {input_csv_path}")
    logger.info(f"Filtering with combined reward threshold >= {combined_reward_threshold}")
    
    # Load the CSV file
    try:
        df = pd.read_csv(input_csv_path)
        logger.info(f"Loaded {len(df)} preference pairs")
    except FileNotFoundError:
        logger.error(f"File not found: {input_csv_path}")
        return {"error": "File not found"}
    except Exception as e:
        logger.error(f"Error loading CSV: {e}")
        return {"error": str(e)}
    
    # Parse chosen reward JSON strings
    def safe_parse_reward(reward_str):
        """Safely parse reward JSON string"""
        if pd.isna(reward_str) or reward_str is None:
            return {}
        try:
            return json.loads(reward_str)
        except json.JSONDecodeError:
            return {}
    
    # Parse reward data
    df['chosen_reward_parsed'] = df['chosen_reward'].apply(safe_parse_reward)
    df['rejected_reward_parsed'] = df['rejected_reward'].apply(safe_parse_reward)
    
    # Extract combined rewards
    df['chosen_combined_reward'] = df['chosen_reward_parsed'].apply(
        lambda x: x.get('combined', 0) if isinstance(x, dict) else 0
    )
    df['rejected_combined_reward'] = df['rejected_reward_parsed'].apply(
        lambda x: x.get('combined', 0) if isinstance(x, dict) else 0
    )
    
    # Filter based on chosen conversation combined reward threshold
    filtered_df = df[df['chosen_combined_reward'] >= combined_reward_threshold].copy()
    
    # Prepare output based on format
    if save_format == "chosen_only":
        # Save only chosen conversations in a simplified format
        output_data = []
        for _, row in filtered_df.iterrows():
            output_row = {
                'id': row['id'],
                'ground_truth': row['ground_truth'],
                'original_conversation': row['original_conversation'],
                'conversation': row['chosen_conversation'],  # Only chosen conversation
                'reward': row['chosen_reward'],
                'combined_reward': row['chosen_combined_reward'],
                'reasoning': row.get('chosen_reasoning', ''),
                'metadata': row['metadata']
            }
            output_data.append(output_row)
        
        output_df = pd.DataFrame(output_data)
        
    else:  # save_format == "preference_pairs"
        # Keep the original preference pair format
        output_df = filtered_df[[
            'id', 'ground_truth', 'original_conversation', 
            'chosen_conversation', 'rejected_conversation',
            'chosen_reward', 'rejected_reward',
            'chosen_reasoning', 'rejected_reasoning',
            'reward_gap', 'metadata'
        ]].copy()
    
    # Create output directory if needed
    output_dir = Path(output_csv_path).parent
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Save filtered results
    output_df.to_csv(output_csv_path, index=False)
    
    # Calculate statistics
    stats = {
        'total_input_pairs': len(df),
        'filtered_pairs': len(filtered_df),
        'filtered_percentage': (len(filtered_df) / len(df) * 100) if len(df) > 0 else 0,
        'threshold_used': combined_reward_threshold,
        'save_format': save_format,
        'output_file': output_csv_path
    }
    
    # Additional statistics
    if len(filtered_df) > 0:
        stats.update({
            'avg_chosen_reward': filtered_df['chosen_combined_reward'].mean(),
            'min_chosen_reward': filtered_df['chosen_combined_reward'].min(),
            'max_chosen_reward': filtered_df['chosen_combined_reward'].max(),
            'avg_rejected_reward': filtered_df['rejected_combined_reward'].mean(),
            'avg_reward_gap': filtered_df['reward_gap'].mean()
        })
    
    # Log results
    logger.info(f"✅ Filtering complete:")
    logger.info(f"   Input pairs: {stats['total_input_pairs']}")
    logger.info(f"   Filtered pairs: {stats['filtered_pairs']} ({stats['filtered_percentage']:.1f}%)")
    logger.info(f"   Saved to: {output_csv_path}")
    
    if len(filtered_df) > 0:
        logger.info(f"📊 Filtered data statistics:")
        logger.info(f"   Avg chosen reward: {stats['avg_chosen_reward']:.3f}")
        logger.info(f"   Reward range: {stats['min_chosen_reward']:.3f} - {stats['max_chosen_reward']:.3f}")
        logger.info(f"   Avg reward gap: {stats['avg_reward_gap']:.3f}")
    else:
        logger.warning(f"⚠️ No conversations met the threshold of {combined_reward_threshold}")
    
    return stats


def filter_by_multiple_criteria(input_csv_path: str,
                               output_csv_path: str,
                               criteria: Dict[str, float],
                               logic: str = "AND",
                               save_format: str = "chosen_only") -> Dict[str, Any]:
    """
    Filter conversations by multiple reward criteria
    
    Args:
        input_csv_path: Path to input CSV
        output_csv_path: Path to save filtered results
        criteria: Dict with metric thresholds, e.g., {'combined': 0.7, 'hit_rate': 1.0, 'interactivity': 0.5}
        logic: "AND" (all criteria must be met) or "OR" (any criteria can be met)
        
    Returns:
        Dictionary with filtering statistics
    """
    
    logger.info(f"Loading preference pairs from: {input_csv_path}")
    logger.info(f"Filtering with criteria: {criteria} (logic: {logic})")
    
    # Load the CSV file
    try:
        df = pd.read_csv(input_csv_path)
        logger.info(f"Loaded {len(df)} preference pairs")
    except FileNotFoundError:
        logger.error(f"File not found: {input_csv_path}")
        return {"error": "File not found"}
    except Exception as e:
        logger.error(f"Error loading CSV: {e}")
        return {"error": str(e)}
    
    # Parse chosen reward JSON strings
    def safe_parse_reward(reward_str):
        """Safely parse reward JSON string"""
        if pd.isna(reward_str) or reward_str is None:
            return {}
        try:
            return json.loads(reward_str)
        except json.JSONDecodeError:
            return {}
    
    # Parse reward data
    df['chosen_reward_parsed'] = df['chosen_reward'].apply(safe_parse_reward)
    df['rejected_reward_parsed'] = df['rejected_reward'].apply(safe_parse_reward)
    
    # Extract combined rewards
    df['chosen_combined_reward'] = df['chosen_reward_parsed'].apply(
        lambda x: x.get('combined', 0) if isinstance(x, dict) else 0
    )
    df['rejected_combined_reward'] = df['rejected_reward_parsed'].apply(
        lambda x: x.get('combined', 0) if isinstance(x, dict) else 0
    )
    
    # Apply filters
    filter_conditions = []
    for metric, threshold in criteria.items():
        df[f'chosen_{metric}'] = df['chosen_reward_parsed'].apply(
            lambda x: x.get(metric, 0) if isinstance(x, dict) else 0
        )
        filter_conditions.append(df[f'chosen_{metric}'] >= threshold)
    
    # Combine conditions
    if logic == "AND":
        final_condition = filter_conditions[0]
        for condition in filter_conditions[1:]:
            final_condition = final_condition & condition
    else:  # OR
        final_condition = filter_conditions[0]
        for condition in filter_conditions[1:]:
            final_condition = final_condition | condition
    
    filtered_df = df[final_condition].copy()
    
    # Prepare output based on format
    if save_format == "chosen_only":
        # Save only chosen conversations in a simplified format
        output_data = []
        for _, row in filtered_df.iterrows():
            output_row = {
                'id': row['id'],
                'ground_truth': row['ground_truth'],
                'original_conversation': row['original_conversation'],
                'conversation': row['chosen_conversation'],  # Only chosen conversation
                'reward': row['chosen_reward'],
                'combined_reward': row['chosen_combined_reward'],
                'reasoning': row.get('chosen_reasoning', ''),
                'metadata': row['metadata']
            }
            output_data.append(output_row)
        
        output_df = pd.DataFrame(output_data)
        
    else:  # save_format == "preference_pairs"
        # Keep the original preference pair format
        output_df = filtered_df[[
            'id', 'ground_truth', 'original_conversation', 
            'chosen_conversation', 'rejected_conversation',
            'chosen_reward', 'rejected_reward',
            'chosen_reasoning', 'rejected_reasoning',
            'reward_gap', 'metadata'
        ]].copy()

    # Save results
    output_dir = Path(output_csv_path).parent
    output_dir.mkdir(parents=True, exist_ok=True)
    output_df.to_csv(output_csv_path, index=False)
    
    # Statistics
    stats = {
        'total_input_pairs': len(df),
        'filtered_pairs': len(output_df),
        'filtered_percentage': (len(output_df) / len(df) * 100) if len(df) > 0 else 0,
        'criteria': criteria,
        'logic': logic
    }
    
    logger.info(f"✅ Multi-criteria filtering complete:")
    logger.info(f"   Filtered: {stats['filtered_pairs']}/{stats['total_input_pairs']} ({stats['filtered_percentage']:.1f}%)")
    
    return stats


# Example usage functions
def main_example():
    """Example usage of the filtering functions"""
    
    # Configuration
    dataset = "redial"
    alg = "vanilla"
    input_file = f"multiturn_test/{alg}/llama3_2_1B/{dataset}/preference_pairs_with_rewards_final.csv"
    
    # Example 1: Filter by combined reward threshold
    output_file_1 = f"multiturn_test/{alg}/llama3_2_1B/{dataset}/high_quality_preference_pairs.csv"
    stats_1 = filter_conversations_by_reward_threshold(
        input_csv_path=input_file,
        output_csv_path=output_file_1,
        combined_reward_threshold=0.7,
        save_format="preference_pairs"
    )
    
    # Example 2: Extract only chosen conversations with high rewards
    output_file_2 = f"multiturn_test/{alg}/llama3_2_1B/{dataset}/high_quality_chosen_conversations.csv"
    stats_2 = filter_conversations_by_reward_threshold(
        input_csv_path=input_file,
        output_csv_path=output_file_2,
        combined_reward_threshold=0.8,
        save_format="chosen_only"
    )
    
    # Example 3: Filter by multiple criteria
    output_file_3 = f"multiturn_test/{alg}/llama3_2_1B/{dataset}/multi_criteria_filtered.csv"
    stats_3 = filter_by_multiple_criteria(
        input_csv_path=input_file,
        output_csv_path=output_file_3,
        criteria={
            'combined': 0.6,
            'hit_rate': 1.0,        # Must mention ground truth
            'interactivity': 0.5    # Must be reasonably interactive
        },
        logic="AND"
    )
    
    return stats_1, stats_2, stats_3


if __name__ == "__main__":
    # Simple usage - filter by combined reward threshold
    dataset = "inspired"
    model = 'llama3-2-1b-instruct'
    
    # File paths
    output_path = f"{dataset}/sft_generated/{model}/train_sft_generated2.csv"
    input_path = f"{dataset}/DPO_offline/{model}/preference_pairs_with_rewards_final.csv"
    
    # stats = filter_conversations_by_reward_threshold(
    #     input_csv_path=input_path,
    #     output_csv_path=output_path,
    #     combined_reward_threshold=1.0,
    #     save_format="chosen_only"  # or "preference_pairs"
    # )
    
    stats = filter_by_multiple_criteria(
        input_csv_path=input_path,
        output_csv_path=output_path,
        criteria={
            'combined': 1.0,
            'hit_rate': 1.0,        
            'interactivity': 0.8    # Must be reasonably interactive
        },
        logic="AND",
        save_format="chosen_only"  # or "preference_pairs"
    )
    
    print(f"Filtering complete! {stats['filtered_pairs']} pairs saved.")

INFO: Loading preference pairs from: inspired/DPO_offline/llama3-2-1b-instruct/preference_pairs_with_rewards_final.csv
INFO: Filtering with criteria: {'combined': 1.0, 'hit_rate': 1.0, 'interactivity': 0.8} (logic: AND)
INFO: Loaded 801 preference pairs
INFO: ✅ Multi-criteria filtering complete:
INFO:    Filtered: 239/801 (29.8%)


Filtering complete! 239 pairs saved.
